# Kernel: dare ordini a migliaia di thread

Il codice del capitolo [«Kernel: dare ordini a migliaia di thread»](https://book.paithon.it/main/GPU/kernel-e-cuda.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torchvision triton

## Kernel: dare ordini a migliaia di thread

[Leggi la pagina](https://book.paithon.it/main/GPU/kernel-e-cuda.html)


### Un kernel in Python: Triton


In [ ]:
import torchimport tritonimport triton.language as tl@triton.jitdef fused_kernel(x_ptr, out_ptr, a, b,                 n_elementi, BLOCK_SIZE: tl.constexpr):    pid = tl.program_id(axis=0)                    # indice del blocco di programma    inizio = pid * BLOCK_SIZE    offsets = inizio + tl.arange(0, BLOCK_SIZE)    # gli indici che questo blocco elabora    mask = offsets < n_elementi                    # non uscire dal bordo dell'array    x = tl.load(x_ptr + offsets, mask=mask)        # UNA lettura dalla memoria    y = tl.maximum(a * x + b, 0.0)                  # a*x + b e poi ReLU, tutto insieme    tl.store(out_ptr + offsets, y, mask=mask)      # UNA scrittura in memoriadef fused_relu(x, a, b):    out = torch.empty_like(x)    n = out.numel()    # quanti blocchi di programma servono per coprire tutti gli elementi    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)    fused_kernel[grid](x, out, a, b, n, BLOCK_SIZE=1024)    return out

## GEMM: la moltiplicazione di matrici, spremuta

[Leggi la pagina](https://book.paithon.it/main/GPU/gemm-e-tensor-core.html)


### Tiling: portare gli ingredienti sul tavolo una volta sola


In [ ]:
import numpy as npdef matmul_a_blocchi(A, B, T=32):    """Moltiplica A (M,K) per B (K,N) lavorando a tessere T×T.    Stesso risultato di A @ B, ma esplicita il riuso: ogni blocco di A    e di B, caricato una volta, serve tutti i prodotti della tessera."""    M, K = A.shape    _, N = B.shape    C = np.zeros((M, N))    for i in range(0, M, T):              # scorre le tessere di righe di C        for j in range(0, N, T):          # scorre le tessere di colonne di C            acc = np.zeros((min(T, M - i), min(T, N - j)))            for k in range(0, K, T):      # somma sui blocchi lungo K                a = A[i:i+T, k:k+T]       # blocco di A -> "shared memory"                b = B[k:k+T, j:j+T]       # blocco di B -> "shared memory"                acc += a @ b              # riuso: un blocco, molti prodotti            C[i:i+T, j:j+T] = acc    return CA = np.random.randn(96, 80)B = np.random.randn(80, 64)print(np.allclose(matmul_a_blocchi(A, B), A @ B))   # True

## Flash Attention: l'attenzione che non spreca memoria

[Leggi la pagina](https://book.paithon.it/main/GPU/flash-attention.html)


### Cosa si guadagna (e cosa costa)


In [ ]:
import torchimport torch.nn.functional as F# Q, K, V: (batch, teste, N, d_k)Q = torch.randn(2, 8, 4096, 64, device="cuda", dtype=torch.float16)K = torch.randn_like(Q)V = torch.randn_like(Q)# PyTorch sceglie da sé il kernel: su GPU recenti, il backend FlashAttention.# is_causal=True applica la maschera causale senza materializzarla.O = F.scaled_dot_product_attention(Q, K, V, is_causal=True)print(O.shape)  # torch.Size([2, 8, 4096, 64])

## Oltre una GPU: parallelismo distribuito

[Leggi la pagina](https://book.paithon.it/main/GPU/parallelismo-distribuito.html)


### Non replicare, spartire: ZeRO e FSDP


In [ ]:
# SCHEMA (come DDP), si lancia con: torchrun --nproc_per_node=4 addestra.pyimport osimport torchimport torch.distributed as distfrom torch.distributed.fsdp import FullyShardedDataParallel as FSDPdist.init_process_group("nccl")rank = int(os.environ["LOCAL_RANK"])torch.cuda.set_device(rank)# invece di REPLICARE il modello (come DDP), FSDP ne SPARTISCE i parametrimodel = FSDP(model, device_id=rank)# training loop IDENTICO: FSDP raduna (all-gather) i pesi di ogni blocco# appena prima di usarlo, e li ri-spartisce subito dopo, in automatico.